# Lamalo Free True-360 Production

Runs the permanent clothing pipeline with FLUX.1-schnell, TRELLIS (preferred), TripoSR fallback, Qwen3-VL visual QA and Blender. No paid generation API is used. Enable a Kaggle GPU before running.


In [ ]:
START_ORDINAL = 1
COUNT = 1
PARITY = 'all'  # all, odd, even
FORCE = False
PUBLISH = True  # requires the existing Virelle DB and R2/S3 secrets


In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/kaggle/working/virellestudios')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/leego972/virellestudios.git',str(repo)], check=True)
os.chdir(repo)
subprocess.run(['bash','scripts/lamalo360/free/bootstrap_kaggle.sh'], check=True)


In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ['HF_TOKEN','DATABASE_URL','AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_S3_BUCKET','AWS_REGION','AWS_S3_ENDPOINT','AWS_S3_PUBLIC_URL']:
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
os.environ['LAMALO360_WORK_ROOT'] = '/kaggle/working/lamalo360-work'
os.environ['TRIPOSR_HOME'] = '/kaggle/working/TripoSR'
os.environ['TRELLIS_HOME'] = '/kaggle/working/TRELLIS'
os.environ['BLENDER_BIN'] = subprocess.check_output(['which','blender'], text=True).strip()
os.environ['LAMALO_FREE_3D_ENGINE'] = 'auto'


In [ ]:
cmd = ['node','scripts/lamalo360/run-batch.mjs','--parity',PARITY,'--count',str(COUNT),'--start',str(START_ORDINAL),'--retry-failed']
if FORCE:
    cmd.append('--force')
if not PUBLISH:
    raise RuntimeError('For review-only runs, execute run-master.mjs with --skip-publish per ordinal; batch publication is fail-closed by design.')
subprocess.run(cmd, check=True)


In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/lamalo360-work-backup','zip','/kaggle/working/lamalo360-work')
print('Completed. Backup:', archive)
